# Quantitative Strategy Analysis: Part 2 - Feature Engineering & Quant Metrics

**Project Portfolio:** Algorithmic Trading Performance Dashboard  
**Objective:** To transform cleaned trade logs into a highly analytical, compounded dataset and calculate institutional-grade quantitative metrics.

### Notebook Objectives:
1. **Asset Mapping:** Translate raw broker symbols (e.g., `XTIUSD.s`) into human-readable asset names (e.g., `CRUDE OIL`).
2. **Rolling Capital Simulation:** Reconstruct the exact account balance at the moment of every trade execution.
   * Account 1 (`ST01`): Starts at **£500**
   * Account 2 (`ST02`): Starts at **£2,700** (with a known £350 withdrawal).
3. **Risk & R-Multiple Calculation:** Apply the strategy's strict 5% risk rule to calculate exact GBP risk and transform raw profits into standardized **R-Multiples**.
4. **Quantitative Evaluation:** Calculate Win Rate, Expectancy, Max Drawdown, Profit Factor, Sharpe Ratio, and Calmar Ratio.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import warnings

warnings.filterwarnings('ignore')

# 1. Load Interim Data
try:
    df_st01 = pd.read_csv('../data/interim/st01_cleaned.csv', parse_dates=['entry_time', 'exit_time'])
    df_st02 = pd.read_csv('../data/interim/st02_cleaned.csv', parse_dates=['entry_time', 'exit_time'])
    print(f"✅ Cleaned data loaded successfully.")
except FileNotFoundError:
    print("❌ Data not found. Please run Notebook 01 first.")

✅ Cleaned data loaded successfully.


## 1. Asset Dictionary Mapping
We map the MT5 symbols to human-readable names for cleaner visualization in the upcoming Streamlit dashboard.

In [2]:
# Standardized mapping dictionary
asset_names = {
    'XTIUSD.s': 'CRUDE OIL', 
    'EURUSD.s': 'EUR/USD', 
    'USDCAD.s': 'USD/CAD',
    'XAUUSD.s': 'GOLD', 
    'SP500.s': 'S&P 500', 
    'GBPUSD.s': 'GBP/USD',
    'USDJPY.s': 'USD/JPY', 
    'AUDUSD.s': 'AUD/USD', 
    'XAGUSD.s': 'SILVER',
    'UK100.s': 'FTSE 100'
}

df_st01['asset_name'] = df_st01['symbol'].map(asset_names).fillna(df_st01['symbol'])
df_st02['asset_name'] = df_st02['symbol'].map(asset_names).fillna(df_st02['symbol'])

## 2. The Compounding & R-Multiple Engine
This function is the core of our quantitative analysis. It iterates through the chronologically sorted trades, applies known deposits/withdrawals, calculates the **5% Risk** based on the capital available *at that exact moment*, and derives the **R-Multiple** (Return on Risk).

In [3]:
def apply_compounding_and_risk(df, starting_capital, risk_percent=0.05, withdrawals=None):
    """
    Simulates the rolling account balance and calculates R-Multiples.
    
    Parameters:
    df (DataFrame): Cleaned and chronologically sorted trade log.
    starting_capital (float): Initial GBP balance.
    risk_percent (float): Fraction of capital risked per trade (default 5%).
    withdrawals (list of dicts): Known cash flows e.g., [{'date': '2026-02-03', 'amount': 350.0}]
    """
    df = df.copy()
    
    capital_at_entry = []
    capital_at_exit = []
    risk_gbp_list = []
    r_multiples = []
    
    current_capital = starting_capital
    withdrawal_index = 0
    
    # Ensure withdrawals are sorted by date if provided
    if withdrawals:
        withdrawals = sorted(withdrawals, key=lambda x: pd.to_datetime(x['date']))
    
    for index, row in df.iterrows():
        trade_exit_time = row['exit_time']
        
        # Apply withdrawal if the trade exit time is after the withdrawal date
        if withdrawals and withdrawal_index < len(withdrawals):
            if trade_exit_time > pd.to_datetime(withdrawals[withdrawal_index]['date']):
                current_capital -= withdrawals[withdrawal_index]['amount']
                withdrawal_index += 1
                
        # 1. Record Capital at Entry & Calculate Risk
        capital_at_entry.append(current_capital)
        risk_gbp = current_capital * risk_percent
        risk_gbp_list.append(risk_gbp)
        
        # 2. Calculate R-Multiple
        profit = row['profit']
        r_val = profit / risk_gbp
        
        # Flag Breakeven trades (where absolute return is effectively zero)
        if abs(r_val) < 0.005:
            r_multiples.append(0.0)
        else:
            r_multiples.append(r_val)
            
        # 3. Update Capital
        current_capital += profit
        capital_at_exit.append(current_capital)
        
    # Append new features to dataframe
    df['capital_at_entry'] = capital_at_entry
    df['capital_at_exit'] = capital_at_exit
    df['risk_gbp'] = risk_gbp_list
    # Convert GBP risk to USD for display/comparison (approx 1.285 exchange rate)
    df['risk_usd'] = df['risk_gbp'] * 1.285 
    df['r_multiple'] = r_multiples
    
    return df, current_capital

# Apply to ST01 (Account 1: £500 Start)
df_fp_processed, final_st01 = apply_compounding_and_risk(df_st01, starting_capital=500.0)

# Apply to ST02 (Account 2: £2700 Start, £350 Withdrawal on 2026-02-03 10:15:03)
ee_withdrawals = [{'date': '2026-02-03 10:15:03', 'amount': 350.0}]
df_ee_processed, final_st02 = apply_compounding_and_risk(df_st02, starting_capital=2700.0, withdrawals=ee_withdrawals)

print(f"ST01 Final Calculated Balance: £{final_st01:,.2f}")
print(f"ST02 Final Calculated Balance: £{final_st02:,.2f}")

ST01 Final Calculated Balance: £6,381.60
ST02 Final Calculated Balance: £4,949.79


## 3. Quantitative Performance Metrics
We will now define a robust metric generation function. This calculates:
* **Win Rate & Profit Factor:** Basic profitability checks.
* **Average R-Multiples:** Proves the structural edge (e.g., losing -1R but winning +2R).
* **Max Drawdown:** The maximum peak-to-trough drop, ensuring the strategy survives proprietary trading firm rules.
* **Sharpe & Calmar Ratios:** Risk-adjusted performance relative to volatility and drawdown.

In [4]:
def calculate_quant_metrics(df, starting_capital):
    """
    Generates a dictionary of institutional performance metrics.
    """
    # 1. Basic Trade Stats
    total_trades = len(df)
    winning_trades = df[df['profit'] > 0]
    losing_trades = df[df['profit'] < 0]
    be_trades = df[df['profit'] == 0]
    
    win_rate = len(winning_trades) / total_trades if total_trades > 0 else 0
    
    # 2. Profit Factor
    gross_profit = winning_trades['profit'].sum()
    gross_loss = abs(losing_trades['profit'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
    
    # 3. R-Multiple Analytics
    avg_win_r = winning_trades['r_multiple'].mean() if not winning_trades.empty else 0
    avg_loss_r = abs(losing_trades['r_multiple'].mean()) if not losing_trades.empty else 0
    expectancy_r = (win_rate * avg_win_r) - ((len(losing_trades)/total_trades) * avg_loss_r)
    
    # 4. Maximum Drawdown (Peak to Trough)
    equity_curve = [starting_capital] + df['capital_at_exit'].tolist()
    peak = equity_curve[0]
    max_dd = 0
    
    for equity in equity_curve:
        if equity > peak:
            peak = equity
        dd = (peak - equity) / peak
        if dd > max_dd:
            max_dd = dd
            
    # 5. Risk-Adjusted Returns (Daily Sharpe & Calmar)
    # Group profits by day to get daily returns
    daily_returns = df.groupby(df['exit_time'].dt.date)['profit'].sum()
    # Create daily equity curve to find daily percentage change
    daily_equity = daily_returns.cumsum() + starting_capital
    daily_pct_change = daily_equity.pct_change().fillna(daily_returns.iloc[0] / starting_capital)
    
    # Assume 252 trading days for annualization
    trading_days = 252
    if daily_pct_change.std() != 0:
        annualized_sharpe = (daily_pct_change.mean() / daily_pct_change.std()) * np.sqrt(trading_days)
    else:
        annualized_sharpe = 0
        
    annualized_return = daily_pct_change.mean() * trading_days
    calmar_ratio = annualized_return / max_dd if max_dd > 0 else np.inf

    # Compile Results
    metrics = {
        "Total Trades": total_trades,
        "Win Rate (%)": round(win_rate * 100, 2),
        "Breakeven Trades": len(be_trades),
        "Profit Factor": round(profit_factor, 2),
        "Max Drawdown (%)": round(max_dd * 100, 2),
        "Average Win (R)": round(avg_win_r, 2),
        "Average Loss (R)": round(avg_loss_r, 2),
        "Expectancy per Trade (R)": round(expectancy_r, 3),
        "Annualized Sharpe Ratio": round(annualized_sharpe, 2),
        "Calmar Ratio": round(calmar_ratio, 2)
    }
    
    return metrics

# Calculate metrics for both accounts
metrics_fp = calculate_quant_metrics(df_fp_processed, 500.0)
metrics_ee = calculate_quant_metrics(df_ee_processed, 2700.0)

# Display as DataFrame for easy reading
df_metrics = pd.DataFrame([metrics_fp, metrics_ee], index=['Account 1 (ST01)', 'Account 2 (ST02)']).T
display(df_metrics)

,Account 1 (ST01),Account 2 (ST02)
Total Trades,86.000,125.000
Win Rate (%),36.050,25.600
Breakeven Trades,6.000,8.000
Profit Factor,1.560,1.160
Max Drawdown (%),49.790,46.380
Average Win (R),3.460,3.060
Average Loss (R),0.780,0.850
Expectancy per Trade (R),0.799,0.207
Annualized Sharpe Ratio,4.580,2.440
Calmar Ratio,49.550,14.780


## 4. Export Final Processed Datasets
The datasets are now enriched with rolling capital, normalized risk, and R-Multiples. We will export these to the `data/processed/` directory. 

These `FP.csv` and `EE.csv` files will serve as the primary data sources for our interactive Plotly/Streamlit Dashboard.

In [5]:
# Create processed directory
os.makedirs('../data/processed', exist_ok=True)

# Format R-Multiple as strings with signs (e.g., "+1.50", "-1.00", "0.00 (BE)") for the dashboard
def format_r(r):
    if r == 0.0:
        return "0.00 (BE)"
    else:
        return f"{r:+.2f}"

df_fp_processed['return_label'] = df_fp_processed['r_multiple'].apply(format_r)
df_ee_processed['return_label'] = df_ee_processed['r_multiple'].apply(format_r)

# Keep only the columns needed for the Dashboard
final_cols = ['entry_time', 'exit_time', 'asset_name', 'symbol', 'type', 'volume', 'entry_price', 'exit_price', 'profit', 'risk_usd', 'r_multiple', 'return_label', 'capital_at_exit']

# Save to processed folder
df_fp_processed[final_cols].to_csv('../data/processed/ST01.csv', index=False)
df_ee_processed[final_cols].to_csv('../data/processed/ST02.csv', index=False)

print("✅ Feature Engineering complete. Final datasets saved to '../data/processed/'")

✅ Feature Engineering complete. Final datasets saved to '../data/processed/'
